In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:
os.listdir()

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc=Document(
    page_content="Hello World!",
    metadata={"source":"https//www.google.com"}

)

In [ ]:
sample_doc

In [ ]:
#For Text data
from langchain_community.document_loaders.text import TextLoader
loader=TextLoader("RAG_text_data.txt",encoding="utf-8")

In [ ]:
document=loader.load()
document

In [ ]:
# for prac

#PDF load

#for simple pdf->pypdfloader
# from langchain_community.document_loaders.pdf import PyPDFLoader
# pdf_loader=PyPDFLoader("RAG_data1.pdf")
# document=pdf_loader.load()
#document


#for complex pdf->PyMuPdfLoader
# from langchain_community.document_loaders.pdf import PyMuPDFLoader
# loader=PyMuPDFLoader("RAG_data1.pdf")
# document=loader.load()


**#INGESTION PIPELINE**

In [ ]:
#DAtA-> Document
#from langchain_core.documents import Document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader



In [ ]:
os.listdir()

In [ ]:
def load_all_pdfs():
  folder_path="data"
  num_docs=0  #i.e no.of docs gng to be created
  all_docs=[]
  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      # get complete pdf path to load the ppdf
      pdf_path=os.path.join(folder_path,filename) # Corrected line
      loader=PyPDFLoader(pdf_path)
      doc=loader.load()

      all_docs.extend(doc)
      num_docs+=1
  print(f"num_docs:{num_docs}")
  print(f"tot pages:{len(all_docs)}")
  return all_docs

In [ ]:
all_pdf_docs=load_all_pdfs()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_text(documents,chunk_size=500,chunk_overlap=50):
  text_splitter=RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
  )
  chunked_docs=text_splitter.split_documents(documents)
  return chunked_docs

In [ ]:
chunks=split_text(all_pdf_docs)

In [ ]:
len(chunks)

In [ ]:
#CHUNKS->EMBEDDINGS
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
  def __init__(self,model_name="all-MiniLM-L6-v2"):
    self.model_name=model_name
    self.model=SentenceTransformer(self.model_name)

  def generate_embedding(self,text):
    embedding=self.model.encode(text,show_progress_bar=True)
    print(f"embedding shape:{embedding.shape}")
    return embedding

In [ ]:
embedding_manager=EmbeddingManager()

In [ ]:
##VECTOR STORE
import chromadb
import uuid
class VectorStoreManager:
  def __init__(self,persistent_directory="vector_store",collection_name="pdf_documents"):
    self.persistent_directory=persistent_directory
    self.Client=None
    self.collection_name=collection_name
    self.collection=None
    self.initialize_store()

  def initialize_store(self):
    os.makedirs(self.persistent_directory,exist_ok=True)
    #create client
    self.client=chromadb.PersistentClient(path=self.persistent_directory)

    self.collection=self.client.get_or_create_collection(
        name=self.persistent_directory, # Consider changing to self.collection_name for clarity
        metadata={"description":"A vector store of pdf docs for RAG"}
    )
    print("succesfully initialized vector store with collection:",self.collection_name)
    print("docs in coll:",self.collection.count())

  def add_documents(self,documents,embeddings):
    if len(documents)!=len(embeddings):
      raise ValueError("Number of documents and embeddings must be equal")
    ids=[]  #unique id
    all_metadata=[]
    documents_text=[]  #original content
    embeddings_list=[]
    for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
      doc_id=f"doc_{uuid.uuid4()}"
      ids.append(doc_id)


      metadata=dict(doc.metadata)
      metadata["index"]=i
      metadata["content_length"]=len(doc.page_content)
      all_metadata.append(metadata)

      documents_text.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

    # Move the add call outside the loop to add all documents at once
    self.collection.add(
        ids=ids,
        metadatas=all_metadata,
        documents=documents_text,
        embeddings=embeddings_list
    )
    print("total docs added in vector store=",len(documents_text))
    print("docs in collection:",self.collection.count())

In [ ]:
vector_store=VectorStoreManager()

In [ ]:
texts=[doc.page_content for doc in chunks]
embedding=embedding_manager.generate_embedding(texts)
vector_store.add_documents(chunks,embedding)

**Retriever Pipeline**

In [ ]:
class RAGRetriever:
  def __init__(self,embedding_manager,vector_store):
    self.embedding_manager=embedding_manager
    self.vector_store=vector_store

  def retrieve(self,query,top_k=5,score_threshold=0.0):
    #query->embedding
    query_embeddings=self.embedding_manager.generate_embedding([query])[0]

    #semantic search
    results=self.vector_store.collection.query(
        query_embeddings=[query_embeddings.tolist()],
        n_results=top_k
    )

    #cosine similarity
    retrieved_docs=[]
    if results["documents"] and results["documents"][0]:
      ids=results["ids"][0]
      metadatas=results["metadatas"][0]
      documents=results["documents"][0]
      distances=results["distances"][0]


      for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):

        similarity_score=1-distance
        if similarity_score>=score_threshold:
          retrieved_docs.append({
              "id":doc_id,
              "document":document,
              "metadata":metadata,
              "distance":distance,
              "similarity_score":similarity_score,
              "rank":i+1
          })
      print("retrieved:",len(retrieved_docs))
    else:
      print("no docs found")
    return retrieved_docs



In [ ]:
rag_retriever=RAGRetriever(embedding_manager,vector_store)

In [ ]:
rag_retriever.retrieve("what is RAG")

**INTEGRATE WITH LLM**

In [ ]:
# Set OPENAI_API_KEY in your environment or local .env file.


In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(
    model="gpt-5.4",
    max_tokens=1024,
    temperature=0.1
)


In [ ]:
#generate our retrieval agumented output
def generate_output(query,rag_retriever,llm,top_k=3):
  results=rag_retriever.retrieve(query,top_k)
  context="\n".join([doc["document"] for doc in results]) if results else ""
  if not context:
    print("we found no relavent context")
  #context+query->prompt
  prompt=f"""use given context to generate the answer for the query context:{context}  query:{query}"""
  response=llm.invoke(prompt)
  return response.content



In [ ]:
answer=generate_output("What is RAG",rag_retriever,llm)

In [ ]:
answer

In [ ]:
# Set GROQ_API_KEY in your environment or local .env file.


In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)


In [ ]:
#generate our retrieval agumented output
def generate_output(query,retriever,llm,top_k=3):
  results=retriever.retrieve(query,top_k)

  context="\n".join([doc["document"] for doc in results]) if results else ""
  if not context:
    print("we found no relavent context for given query")

  #context+query
  promt= f"""use given context to generate the answer for the query context:{context} query:{query}"""

  response=llm.invoke([promt.format(context=context,query=query)])  ## gqor is expecting a list

  return response.content




In [ ]:
answer=generate_output("what is encoder-decoder?",rag_retriever,llm)

In [ ]:
print(answer)

In [ ]:
#claude
